In [44]:
## Running the Data Extraction File
%run data_extraction.ipynb

<class 'pandas.DataFrame'>
DatetimeIndex: 1337 entries, 2021-01-04 to 2026-04-30
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   BUFR    1337 non-null   float64
 1   IJR     1337 non-null   float64
 2   SMH     1337 non-null   float64
 3   SPY     1337 non-null   float64
 4   UGL     1337 non-null   float64
 5   VO      1337 non-null   float64
dtypes: float64(6)
memory usage: 73.1 KB


In [45]:
## Calculate Daily Asset Returns
def calculate_asset_returns(prices_wide_df):
    returns_wide = prices_wide_df.pct_change().dropna(how="all")
    returns_wide.index.name = "Date"
    return returns_wide

returns_wide = calculate_asset_returns(prices_wide)

## Outputting the Daily Asset Return
returns_wide

Ticker,BUFR,IJR,SMH,SPY,UGL,VO
Date,,,,,,
2021-01-05,0.007754,0.021309,0.017500,0.006887,0.005044,0.009522
2021-01-06,0.010026,0.048612,-0.003054,0.005979,-0.035271,0.016628
2021-01-07,0.000000,0.010564,0.041259,0.014857,-0.005202,0.014347
2021-01-08,-0.000462,-0.008221,-0.003590,0.005698,-0.070453,0.004526
2021-01-11,-0.001940,0.004605,0.014978,-0.006741,-0.001875,-0.002629
...,...,...,...,...,...,...
2026-04-24,0.002821,0.005385,0.051032,0.007749,0.008316,-0.000911
2026-04-27,0.001041,0.001908,-0.000355,0.001723,-0.015200,-0.001955
2026-04-28,-0.002445,-0.005639,-0.029728,-0.004866,-0.036617,-0.007443


In [46]:
## Note. Long Format Version
def reshape_returns_long(returns_wide_df):
    returns_long = (
        returns_wide_df
        .reset_index()
        .melt(id_vars="Date", var_name="Ticker", value_name="Daily_Return")
        .sort_values(["Ticker", "Date"])
        .reset_index(drop=True)
    )
    return returns_long

returns_long = reshape_returns_long(returns_wide)

## Outputting the New Return Format
returns_long.head()

,Date,Ticker,Daily_Return
0,2021-01-05,BUFR,0.007754
1,2021-01-06,BUFR,0.010026
2,2021-01-07,BUFR,0.000000
3,2021-01-08,BUFR,-0.000462
4,2021-01-11,BUFR,-0.001940


In [47]:
## Creating a Target Weights Table
weights_df = pd.DataFrame(
    list(TARGET_WEIGHTS.items()),
    columns=["Ticker", "Target_Weight"]
)

## Checking the Weights Table
weights_df

,Ticker,Target_Weight
0,SMH,0.25
1,VO,0.15
2,IJR,0.15
3,BUFR,0.15
4,UGL,0.15
5,SPY,0.15


In [48]:
## Calculating Portfolio Daily Returns
def calculate_portfolio_returns_constant_weight(returns_wide_df, target_weights):
    weights = pd.Series(target_weights)
    weights = weights[returns_wide_df.columns]
    
    portfolio_returns = returns_wide_df.mul(weights, axis=1).sum(axis=1)
    portfolio_returns = portfolio_returns.to_frame(name="Portfolio_Return")
    
    return portfolio_returns

## Executing and Outputting the Data
portfolio_returns = calculate_portfolio_returns_constant_weight(returns_wide, TARGET_WEIGHTS)
portfolio_returns.head()

portfolio_returns["Benchmark_Return"] = returns_wide[BENCHMARK]
portfolio_returns.head()

,Portfolio_Return,Benchmark_Return
Date,,
2021-01-05,0.011952,0.017500
2021-01-06,0.006133,-0.003054
2021-01-07,0.015500,0.041259
2021-01-08,-0.011234,-0.003590
2021-01-11,0.002457,0.014978


In [49]:
## Calculating Cumulative Growth and Portfolio Value
def calculate_growth_series(return_series, initial_value=10000):
    growth_index = (1 + return_series).cumprod()
    portfolio_value = initial_value * growth_index
    return growth_index, portfolio_value

portfolio_returns["Portfolio_Growth_Index"], portfolio_returns["Portfolio_Value"] = calculate_growth_series(
    portfolio_returns["Portfolio_Return"],
    INITIAL_PORTFOLIO_VALUE
)

portfolio_returns["Benchmark_Growth_Index"], portfolio_returns["Benchmark_Value"] = calculate_growth_series(
    portfolio_returns["Benchmark_Return"],
    INITIAL_PORTFOLIO_VALUE
)

## Outputting the Data
portfolio_returns

,Portfolio_Return,Benchmark_Return,Portfolio_Growth_Index,Portfolio_Value,Benchmark_Growth_Index,Benchmark_Value
Date,,,,,,
2021-01-05,0.011952,0.017500,1.011952,10119.523917,1.017500,10174.998081
2021-01-06,0.006133,-0.003054,1.018158,10181.582968,1.014393,10143.927551
2021-01-07,0.015500,0.041259,1.033940,10339.395277,1.056246,10562.459240
2021-01-08,-0.011234,-0.003590,1.022324,10223.238826,1.052454,10524.536522
2021-01-11,0.002457,0.014978,1.024836,10248.362427,1.068217,10682.171321
...,...,...,...,...,...,...
2026-04-24,0.016262,0.051032,2.707045,27070.448428,4.769808,47698.081819
2026-04-27,-0.001961,-0.000355,2.701735,27017.350074,4.768113,47681.129553
2026-04-28,-0.015983,-0.029728,2.658552,26585.519547,4.626367,46263.672399
